In [1]:
import numpy as np
import math
from collections import Counter

from pyqubo import Array
import neal


# Budget Optimization as a QUBO #    

- Variables:    $  x[i] ∈ {0,1} $ -> buy item i or not    

- Objective (maximise items):   maximise $ \sum_i x[i] $    

- QUBO minimises, so we minimise:   $ -\sum_i x[i]$   
   
- Constraint (budget):   $ \sum_i cost[i] * x[i]  \leq B $    
   
     
**Trick to make $\leq$ a quadratic penalty:**  
    
introduce a non-negative "slack" integer s such that:   $ \sum_i cost[i] * x[i] + s = B$   
  
with s encoded in binary using extra bits $y[k] \in {0,1}$.   
  
If $\sum_i cost[i] * x[i] > B$, equality cannot be satisfied (since $s \geq 0$), so the penalty becomes large.

1) Declare N and generate random costs (floats)

In [17]:
N = 10
rng = np.random.default_rng(7)  # seed for reproducibility 
costs = np.round(rng.uniform(1.0, 10.0, size=N), 2)  # N floats in [1,10]

print("Costs:")
print(costs)

Costs:
[6.63 9.07 7.98 3.03 3.7  8.86 1.05 8.39 8.17 5.21]


2) Decide a budget B (float)    
 (here: about 50% of total cost, but you can set it manually)

In [18]:
B = float(np.round(0.50 * costs.sum(), 2))
print("\nTotal cost sum =", float(np.round(costs.sum(), 2)))
print("Budget B =", B)


Total cost sum = 62.09
Budget B = 31.04


3) Convert to integers (scale by 100 to keep cents)

In [19]:
SCALE = 100
costs_int = (costs * SCALE).astype(int)
B_int = int(round(B * SCALE))

4) Optional constraint: "buy only items with an odd index"    
  - Python indices: $0,1,2,...$ so odd indices are $1,3,5,...$    
  - We enforce it by adding a linear penalty to even-index $x[i]$    

In [20]:
odd_only = True  # set True to activate the optional constraint

5) Build the QUBO with PyQUBO

In [21]:
# Decision variables (buy/not buy)
x = Array.create("x", shape=N, vartype="BINARY")

# Slack bits y encode s in [0, 2^m - 1]
# Need max slack at least B_int (when buying nothing, s = B_int)
m = int(math.ceil(math.log2(B_int + 1)))
y = Array.create("y", shape=m, vartype="BINARY")

# Expressions for total cost and slack
total_cost = sum(costs_int[i] * x[i] for i in range(N))
slack = sum((2 ** k) * y[k] for k in range(m))  # s = Σ 2^k y[k]

# Penalty strength (make constraint important vs objective)
A = 10  # you can increase if you see budget violations

# Constraint penalty: A * (total_cost + slack - B_int)^2
constraint = A * (total_cost + slack - B_int) ** 2

# Objective: maximise Σ x[i]  -> minimise -Σ x[i]
objective = -sum(x[i] for i in range(N))

# Optional odd-index constraint (penalise even indices)
odd_penalty = 0
if odd_only:
    P_even = 5  # must be > 1 so choosing an even index is never beneficial
    odd_penalty = P_even * sum(x[i] for i in range(N) if i % 2 == 0)

In [22]:
# Full QUBO energy to minimise
H = constraint + objective + odd_penalty

In [23]:
# Compile to QUBO and solve with D-Wave's simulated annealing (neal)
model = H.compile()
Q, offset = model.to_qubo()

sampler = neal.SimulatedAnnealingSampler()
sampleset = sampler.sample_qubo(Q, num_reads=500)

decoded = model.decode_sampleset(sampleset)
best = min(decoded, key=lambda s: s.energy)

6) Extract best solution

In [24]:
best_x = np.array([best.sample[f"x[{i}]"] for i in range(N)], dtype=int)

used_budget_int = int(np.dot(costs_int, best_x))
used_budget = used_budget_int / SCALE

num_items = int(best_x.sum())

print("\n================ BEST SOLUTION ================")
print("Solution vector x:", best_x.tolist())
print("Number of items  :", num_items)
print("Used budget      :", used_budget)
print("Total budget     :", B)
print("Feasible?        :", used_budget_int <= B_int)


================ BEST SOLUTION ================
Solution vector x: [0, 1, 0, 1, 0, 0, 0, 1, 0, 1]
Number of items  : 4
Used budget      : 25.7
Total budget     : 31.04
Feasible?        : True


In [25]:
if odd_only:
    even_selected = [i for i in range(N) if (i % 2 == 0 and best_x[i] == 1)]
    print("Even indices selected (should be empty):", even_selected)

Even indices selected (should be empty): []
